<a href="https://colab.research.google.com/github/keyvanmahmoudi/Python_Course/blob/main/solutions/Session16_final_project_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl">

# پاسخ تمرین‌های جلسهٔ ۱۶: پروژهٔ پایانی و ادامهٔ مسیر

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session16_final_project.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

</div>

<div dir="rtl">

سلول زیر کلاس‌های پروژه و کتابخانهٔ نمونهٔ نوت‌بوک جلسه را دوباره می‌سازد تا پاسخ‌ها مستقل اجرا شوند.

</div>

In [1]:
class LibraryError(Exception):
    """Raised for invalid library operations."""


class Book:
    def __init__(self, book_id, title, author, year, borrowed_by=None):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.year = year
        self.borrowed_by = borrowed_by      # member id, or None if available

    @property
    def available(self):
        return self.borrowed_by is None

    def to_dict(self):
        return {"book_id": self.book_id, "title": self.title, "author": self.author,
                "year": self.year, "borrowed_by": self.borrowed_by}

    @classmethod
    def from_dict(cls, data):
        return cls(**data)

    def __str__(self):
        status = "available" if self.available else f"borrowed by {self.borrowed_by}"
        return f"[{self.book_id}] {self.title} - {self.author} ({self.year}) - {status}"


class Member:
    MAX_BOOKS = 3

    def __init__(self, member_id, name, books=None):
        self.member_id = member_id
        self.name = name
        self.books = books if books is not None else []   # list of book ids

    def can_borrow(self):
        return len(self.books) < self.MAX_BOOKS

    def to_dict(self):
        return {"member_id": self.member_id, "name": self.name, "books": self.books}

    @classmethod
    def from_dict(cls, data):
        return cls(**data)

    def __str__(self):
        return f"{self.name} ({self.member_id}) - {len(self.books)} book(s)"


import json
from collections import Counter


class Library:
    def __init__(self):
        self.books = {}
        self.members = {}

    # ---------- adding ----------
    def add_book(self, book):
        if book.book_id in self.books:
            raise LibraryError(f"book id {book.book_id} already exists")
        self.books[book.book_id] = book

    def add_member(self, member):
        if member.member_id in self.members:
            raise LibraryError(f"member id {member.member_id} already exists")
        self.members[member.member_id] = member

    # ---------- lookups ----------
    def _get_book(self, book_id):
        try:
            return self.books[book_id]
        except KeyError:
            raise LibraryError(f"no book with id {book_id}") from None

    def _get_member(self, member_id):
        try:
            return self.members[member_id]
        except KeyError:
            raise LibraryError(f"no member with id {member_id}") from None

    # ---------- operations ----------
    def borrow(self, member_id, book_id):
        member = self._get_member(member_id)
        book = self._get_book(book_id)
        if not book.available:
            raise LibraryError(f"'{book.title}' is already borrowed")
        if not member.can_borrow():
            raise LibraryError(f"{member.name} already has {Member.MAX_BOOKS} books")
        book.borrowed_by = member_id
        member.books.append(book_id)

    def give_back(self, member_id, book_id):
        member = self._get_member(member_id)
        book = self._get_book(book_id)
        if book_id not in member.books:
            raise LibraryError(f"{member.name} has not borrowed '{book.title}'")
        book.borrowed_by = None
        member.books.remove(book_id)

    def search(self, text):
        text = text.lower()
        return [b for b in self.books.values()
                if text in b.title.lower() or text in b.author.lower()]

    # ---------- reports ----------
    def available_books(self):
        return [b for b in self.books.values() if b.available]

    def borrowed_books(self):
        return [b for b in self.books.values() if not b.available]

    def books_per_author(self):
        return Counter(b.author for b in self.books.values())

    # ---------- persistence ----------
    def save(self, path):
        data = {"books": [b.to_dict() for b in self.books.values()],
                "members": [m.to_dict() for m in self.members.values()]}
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    @classmethod
    def load(cls, path):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        library = cls()
        for item in data["books"]:
            library.add_book(Book.from_dict(item))
        for item in data["members"]:
            library.add_member(Member.from_dict(item))
        return library


library = Library()

books = [
    ("B1", "Python Crash Course", "Eric Matthes", 2023),
    ("B2", "Fluent Python", "Luciano Ramalho", 2022),
    ("B3", "Automate the Boring Stuff", "Al Sweigart", 2019),
    ("B4", "Think Python", "Allen Downey", 2024),
    ("B5", "Invent Your Own Games", "Al Sweigart", 2016),
]
for args in books:
    library.add_book(Book(*args))

for member_id, name in [("M1", "Sara"), ("M2", "Ali")]:
    library.add_member(Member(member_id, name))

print(len(library.books), "books,", len(library.members), "members")


library.borrow("M1", "B1")
library.borrow("M1", "B3")


5 books, 2 members


<div dir="rtl">

### تمرین ۱

به کلاس `Library` متد `remove_book(book_id)` اضافه کنید که اگر کتاب امانت رفته بود، اجازهٔ حذف ندهد و `LibraryError` ایجاد کند.

</div>

In [2]:
def remove_book(self, book_id):
    book = self._get_book(book_id)
    if not book.available:
        raise LibraryError(f"cannot remove '{book.title}': it is borrowed")
    del self.books[book_id]

Library.remove_book = remove_book

try:
    library.remove_book("B1")
except LibraryError as e:
    print("ERROR:", e)
library.remove_book("B5")
print(sorted(library.books))

ERROR: cannot remove 'Python Crash Course': it is borrowed
['B1', 'B2', 'B3', 'B4']


<div dir="rtl">

### تمرین ۲

متد `oldest_books(n)` بنویسید که `n` کتاب قدیمی‌تر را (بر اساس سال انتشار) برگرداند.

</div>

In [3]:
def oldest_books(self, n=3):
    return sorted(self.books.values(), key=lambda b: b.year)[:n]

Library.oldest_books = oldest_books
for book in library.oldest_books(2):
    print(book)

[B3] Automate the Boring Stuff - Al Sweigart (2019) - borrowed by M1
[B2] Fluent Python - Luciano Ramalho (2022) - available


<div dir="rtl">

### تمرین ۳

متد `export_csv(path)` بنویسید که فهرست همهٔ کتاب‌ها را با ستون‌های `book_id`، `title`، `author`، `year` و `available` در فایل CSV ذخیره کند.

</div>

In [4]:
import csv

def export_csv(self, path):
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["book_id", "title", "author", "year", "available"])
        for b in self.books.values():
            writer.writerow([b.book_id, b.title, b.author, b.year, b.available])

Library.export_csv = export_csv
library.export_csv("books.csv")
with open("books.csv", encoding="utf-8") as f:
    print(f.read())

book_id,title,author,year,available
B1,Python Crash Course,Eric Matthes,2023,False
B2,Fluent Python,Luciano Ramalho,2022,True
B3,Automate the Boring Stuff,Al Sweigart,2019,False
B4,Think Python,Allen Downey,2024,True



<div dir="rtl">

### تمرین ۴

ثبت تاریخ امانت را اضافه کنید: هنگام امانت تاریخ را ذخیره کنید و تابعی بنویسید که با گرفتن تاریخ بازگشت، جریمهٔ دیرکرد (روزی ۵۰۰۰ تومان پس از ۱۴ روز) را حساب کند.

</div>

In [5]:
from datetime import date

LOAN_DAYS = 14
DAILY_FINE = 5000

def late_fee(borrow_date, return_date):
    days = (return_date - borrow_date).days
    late_days = max(0, days - LOAN_DAYS)
    return late_days * DAILY_FINE

print(late_fee(date(2026, 11, 1), date(2026, 11, 10)))
print(late_fee(date(2026, 11, 1), date(2026, 11, 25)))

0
50000
